# Notebook 2 — Phase 2: 5-Config Retrieval Ablation Study (ONNX Laptop vs. BGE GPU)

**Kaggle Settings required (right sidebar):**
- Accelerator: **GPU T4 x2** (or **GPU P100**)
- Internet: **On**
- Add-ons → Secrets: tick **`QDRANT_URL`** and **`QDRANT_API_KEY`**

Evaluates 35 RAG queries (`factual_rag` + `multi_hop_rag`) across 5 retrieval pipelines using Dynamic Evidence Matching (Recall@5, MRR@5, nDCG@5, Latency).

In [ ]:
# Cell 1: Clone or pull the latest repo from GitHub
from pathlib import Path

REPO_URL = "https://github.com/Nody3112V/AI-papers-agentic-RAG.git"
REPO_DIR = Path("/kaggle/working/AI-papers-agentic-RAG")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
# Cell 2: Install retrieval & heavy reranker dependencies for Kaggle GPU
!pip install -q qdrant-client fastembed pyyaml pydantic-settings sentence-transformers

In [ ]:
# Cell 3: Load Qdrant Cloud secrets securely from Kaggle Secrets
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["QDRANT_MODE"] = "cloud"
os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
print("Qdrant Cloud secrets loaded.")

In [ ]:
# Cell 4: Run all 5 retrieval configurations on 35 RAG queries (takes ~1-2 minutes)
!python -u eval/eval_retrieval.py --device cuda --configs all --top-k 5 --out-dir eval/reports

In [ ]:
# Cell 5: Package ablation JSON + Markdown reports for download to Laptop
import shutil
from pathlib import Path

archive = shutil.make_archive("/kaggle/working/phase2_ablation_reports", "zip", "eval/reports")
print("Saved report archive:", archive)
for md in sorted(Path("eval/reports").glob("*.md")):
    print("\n" + md.read_text(encoding="utf-8"))